In [213]:
from ruwordnet import RuWordNet
import pymorphy3
import re

wn = RuWordNet()
morph = pymorphy3.MorphAnalyzer()

In [214]:
def preprocess(text):
    import re
    words = re.findall(r'\w+', text.lower())
    
    lemmas = []
    for w in words:
        p = morph.parse(w)[0]
        
        # фильтр частей речи
        if p.tag.POS not in {'PREP', 'CONJ', 'PRCL'}:
            lemmas.append(p.normal_form)

    return set(lemmas)

In [215]:
def expanded(words):
    expanded_set = set(words)

    for w in words:
        senses = wn.get_senses(w)
        
        for sense in senses:
            synset = sense.synset

            for s in synset.senses:
                lemmas = preprocess(s.name)
                expanded_set.update(lemmas)

            for h in synset.hypernyms[:1]:
                for s in h.senses:
                    lemmas = preprocess(s.name)
                    expanded_set.update(lemmas)

            for h in synset.hyponyms[:1]:
                for s in h.senses:
                    lemmas = preprocess(s.name)
                    expanded_set.update(lemmas)

    return expanded_set

In [216]:
def lesk(sentence, word):
    context = preprocess(sentence)
    context = expanded(context)
    
    senses = wn.get_senses(word)
    
    best_synset = None
    best_score = 0

    for sense in senses:
        synset = sense.synset
        
        text = synset.definition or ""
        
        for s in synset.senses:
            text += " " + s.name
        
        for h in synset.hypernyms[:1]:
            for s in h.senses:
                text += " " + s.name

        for h in synset.hyponyms[:1]:
            for s in h.senses:
                text += " " + s.name

        signature = preprocess(text)
        
        intersection = len(context & signature)

        if len(signature) == 0:
            score = 0
        else:
            score = intersection / len(signature)

        if score  > best_score :
            best_score = score 
            best_synset = synset

    return best_synset

In [217]:
def get_relations(synset):
    synonyms = []
    for sense in synset.senses:
        synonyms.append(sense.name)

    hypernyms = []
    for h in synset.hypernyms[:1]:
        for s in h.senses:
            hypernyms.append(s.name)

    hyponyms = []
    for h in synset.hyponyms[:1]:
        hyponyms.append(h.title)

    return synonyms, hypernyms, hyponyms

In [218]:
sentence = "Я получил хорошую оценку за экзамен"
word = "оценка"

synset = lesk(sentence, word)

print("Выбранное значение:")
print(synset.definition)

synonyms, hypernyms, hyponyms = get_relations(synset)

print("\nСинонимы:", synonyms)
print("Гиперонимы:", hypernyms)
print("Гипонимы:", hyponyms)

Выбранное значение:
отметка, выставляемая в учебных заведениях

Синонимы: ['ОТМЕТКА', 'ОЦЕНКА', 'УЧЕБНАЯ ОТМЕТКА', 'ОЦЕНКА УЧЕНИКА', 'ОЦЕНКА УЧАЩЕГОСЯ', 'УЧЕБНАЯ ОЦЕНКА']
Гиперонимы: ['ОЦЕНКА', 'ОЦЕНОЧНАЯ ХАРАКТЕРИСТИКА']
Гипонимы: ['ЧЕТВЕРКА (ШКОЛЬНАЯ ОТМЕТКА)']
